# 70 — M6 multi-step certificate

M6 専用ノートブックです。凍結 LOCK（ノルム・cutoff・チャネル・source・error ledger・schema）の下で
`final_certificate/` を組み立て、独立 verifier を通し、`M6_acceptance.json` を書きます。

連続極限・質量ギャップは主張しません。`CERTIFIED` は \(q_{cert}<1\) のときだけです。
親 M5 が `NOT_CERTIFIED` の場合、同一スキーム継承では検証済み非収縮完了になり得ます。


In [ ]:
# 必ず最初に pytest を用意する。
import importlib.util, subprocess, sys
if importlib.util.find_spec('pytest') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pytest>=8'])
import pytest
print('pytest:', pytest.__version__)


In [ ]:
from pathlib import Path
import os, sys

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
    Path('/storage') / BUNDLE_NAME,
]
# Prefer a root that actually contains src/ and tests/ (ignore stale env pointing at notebooks/).
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'validated_4d_su2_rg_codex_design.md').is_file()
    and (p.expanduser() / 'src').is_dir()
    and (p.expanduser() / 'tests').is_dir()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError(
        'M6 project root was not found. '
        'Clone/pull github_test into /notebooks/validated_4d_su2_rg_codex_bundle '
        'and unset a bad VALIDATED_RG_PROJECT_ROOT if it points at notebooks/.'
    )
PERSIST_ROOT = Path(os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
os.environ.setdefault('VALIDATED_RG_PERSIST_ACK', 'I_CONFIRM_THIS_PATH_IS_PERSISTENT')
os.environ.setdefault('VALIDATED_RG_M5_RUN_ID', 'M5-20260720T051020Z-c3800fceaa80')
os.environ.setdefault('VALIDATED_RG_M6_RUN_ID', 'M6-20260720T061700Z-7c4e91a2b850')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT)
print('PERSIST_ROOT =', PERSIST_ROOT)
print('M5', os.environ['VALIDATED_RG_M5_RUN_ID'])
print('M6', os.environ['VALIDATED_RG_M6_RUN_ID'])


In [ ]:
from pathlib import Path
import hashlib, json, os

PREVIOUS, CURRENT, RUN_ENV = 'M5', 'M6', 'VALIDATED_RG_M5_RUN_ID'
run_id = os.environ.get(RUN_ENV)
if not run_id:
    raise RuntimeError(f'{RUN_ENV} を受理済み M5 run ID に設定してください。')
if Path(run_id).name != run_id or not run_id.startswith('M5-'):
    raise RuntimeError('M5 run ID の形式が不正です。')
persist_root = Path(os.environ['VALIDATED_RG_PERSIST_ROOT']).expanduser().resolve()
acceptance_path = persist_root / 'runs' / run_id / 'reports' / 'M5_acceptance.json'
if not acceptance_path.is_file():
    raise RuntimeError(f'M5 acceptance artifact がありません: {acceptance_path}')
acceptance = json.loads(acceptance_path.read_text(encoding='utf-8'))
required = {'milestone': 'M5', 'phase': 'M5_COMPLETE', 'status': 'PASS'}
if any(acceptance.get(k) != v for k, v in required.items()):
    raise RuntimeError('M5 acceptance artifact の identity/status が不正です。')
allowed_cert = {'NOT_CERTIFIED', 'ONE_STEP_CERTIFIED'}
if acceptance.get('certification_status') not in allowed_cert:
    raise RuntimeError(f'Unexpected M5 certification_status: {acceptance.get("certification_status")!r}')
if acceptance.get('accepted_for_next_milestone') != 'M6':
    raise RuntimeError('M5 acceptance is not marked accepted_for_next_milestone=M6.')
STAGE_GATE = {
    'current_milestone': CURRENT,
    'predecessor': PREVIOUS,
    'status': 'PREDECESSOR_ACCEPTED',
    'acceptance_path': str(acceptance_path),
    'acceptance_sha256': hashlib.sha256(acceptance_path.read_bytes()).hexdigest(),
    'm5_certification_status': acceptance.get('certification_status'),
    'm5_run_id': run_id,
}
STAGE_GATE


In [ ]:
# Notebook cwd is often .../notebooks; always resolve tests from PROJECT_ROOT.
import os
from pathlib import Path
import pytest

os.chdir(PROJECT_ROOT)
test_files = [
    PROJECT_ROOT / 'tests/test_m6_orchestrator.py',
    PROJECT_ROOT / 'tests/test_one_step_certificate.py',
    PROJECT_ROOT / 'tests/test_independent_one_step_verifier.py',
]
missing = [str(path) for path in test_files if not path.is_file()]
if missing:
    raise RuntimeError(
        'M6 test files are missing from PROJECT_ROOT. '
        'git pull latest main, then Kernel → Restart & Run All. '
        f'Missing: {missing}'
    )
rc = pytest.main([
    '-q',
    f'--rootdir={PROJECT_ROOT}',
    *[str(path) for path in test_files],
])
if rc != 0:
    raise RuntimeError(f'M6 required CPU suite failed with code {rc}')
print('M6 required CPU suite: PASS')
print('cwd =', Path.cwd())
print('PROJECT_ROOT =', PROJECT_ROOT)


In [ ]:
from pathlib import Path
import os, sys

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
    Path('/storage') / BUNDLE_NAME,
]
# Prefer a root that actually contains src/ and tests/ (ignore stale env pointing at notebooks/).
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'validated_4d_su2_rg_codex_design.md').is_file()
    and (p.expanduser() / 'src').is_dir()
    and (p.expanduser() / 'tests').is_dir()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError(
        'M6 project root was not found. '
        'Clone/pull github_test into /notebooks/validated_4d_su2_rg_codex_bundle '
        'and unset a bad VALIDATED_RG_PROJECT_ROOT if it points at notebooks/.'
    )
PERSIST_ROOT = Path(os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
os.environ.setdefault('VALIDATED_RG_PERSIST_ACK', 'I_CONFIRM_THIS_PATH_IS_PERSISTENT')
os.environ.setdefault('VALIDATED_RG_M5_RUN_ID', 'M5-20260720T051020Z-c3800fceaa80')
os.environ.setdefault('VALIDATED_RG_M6_RUN_ID', 'M6-20260720T061700Z-7c4e91a2b850')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT)
print('PERSIST_ROOT =', PERSIST_ROOT)
print('M5', os.environ['VALIDATED_RG_M5_RUN_ID'])
print('M6', os.environ['VALIDATED_RG_M6_RUN_ID'])


## M6 実装境界

- LOCK: Frobenius、`j2_max=1`、χ=16、left-associated fusion、5 source class、E1–E12 ledger。
- 成果物: `artifacts/final_certificate/`、`reports/M6_report.json`、`reports/M6_acceptance.json`。
- 親 M5 one-step influence を singleton family inclusion の下で最終 majorant として継承。
- `CERTIFIED` は \(q_{cert}<1\) のときのみ。現状親が非収縮なら `NOT_CERTIFIED` の検証済み完了になり得ます。
- continuum / mass gap は非主張。scheme 変更時は LOCK 変更管理と ledger 再開放が必要です。
